# Stage 00b — Figure Matching

**What this notebook does:**  
Reads the *Brief Description of the Drawings* text from the PatSeer Excel and
assigns a figure label to every downloaded image using **pure positional
matching** — no OCR, no external models.

The logic is:
1. Load all files for a patent from `raw/{patent_id}/` in order: `img*` → `D*` → `FAT*`
2. For `D` and `FAT` files, detect horizontal/vertical whitespace bands and split
   pages that contain multiple figures
3. Build one global ordered crop list
4. Parse the description text → ordered figure key list (FIG. 1, FIG. 2A, …)
5. If **crop count == description count** → positional 1-to-1 match → `_F` names
6. If **counts differ** → flag everything `_Fu` (needs human review via Stage 01)

**Output naming:**

| Before | After | Meaning |
|--------|-------|---------|
| `US…_img003.png` | `US…_img003_F002B.png` | matched to FIG. 2B |
| `US…_img003.png` | `US…_img003_Fu001.png` | count mismatch — needs review |
| `US…_D00005.png` | `US…_D00005_crop01_F001.png` | first crop of a split D sheet |
| `US…_D00005.png` | kept as-is | original D file always preserved |

**Where it fits in the pipeline:**
```
00a  (PatSeer download)
 ↓
00b  ←  YOU ARE HERE  (positional matching → _F / _Fu labels)
 ↓
01   (optional OCR fallback for _Fu files that need review)
 ↓
02   (pad + resize to 518×518)
```

---

| Cell | What it does |
|------|--------------|
| 1 | Load config and Excel index (reads `description_of_drawings` column) |
| 2 | Dry-run on one patent — shows split detection, crop list, rename preview; **no files changed** |
| 3 | Full run — match and rename all patents in `raw/` |
| 4 | Print matching report (_F count, _Fu count, originals kept) |

In [1]:
import sys, importlib.util, re
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
src_dir   = repo_root / "src"

_cl_path = src_dir / "config_loader.py"
_cl_spec = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod  = importlib.util.module_from_spec(_cl_spec)
sys.modules["config_loader"] = _cl_mod
_cl_spec.loader.exec_module(_cl_mod)
load_config = _cl_mod.load_config

print(f"config_loader loaded from: {_cl_path}")

for p in [str(repo_root), str(src_dir)]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(src_dir))

from ocr_labeler import ocr_and_rename_crops
from extractor import load_patseer_excel

cfg         = load_config()
excel_index = load_patseer_excel(cfg["paths"]["patseer_excel"])
print(f"Loaded {len(excel_index)} patents from Excel.")


def _patent_core(patent_id: str) -> str:
    """Strip country code + kind code; pad compact 6-digit US serial to 7."""
    m = re.match(r"^[A-Z]{2,}(\d+)", patent_id, re.IGNORECASE)
    if not m:
        return patent_id
    num = m.group(1)
    m2 = re.match(r"^((?:19|20)\d{2})(\d{6})$", num)
    if m2:
        num = m2.group(1) + "0" + m2.group(2)
    return num


def _build_folder_map(raw_dir: Path) -> dict:
    """core → actual folder Path, so any patent ID variant resolves correctly."""
    return {_patent_core(d.name): d for d in raw_dir.iterdir() if d.is_dir()}


def _collect_figures(img_dir: Path, patent_id: str) -> list[Path]:
    """Unprocessed figures in document order: img* → D* → FAT*. Skips already-labeled files."""
    already = re.compile(r"_(?:F\d{3}|Fu\d{3})", re.IGNORECASE)

    def _num(p, tag):
        part = p.stem.split(tag)[-1]
        m = re.search(r"\d+", part)
        return int(m.group()) if m else 0

    def _keep(paths, tag):
        return sorted([p for p in paths if not already.search(p.stem)],
                      key=lambda p: _num(p, tag))

    return (_keep(img_dir.glob(f"{patent_id}_img*.png"), "_img") +
            _keep(img_dir.glob(f"{patent_id}_D*.png"),   "_D")   +
            _keep(img_dir.glob(f"{patent_id}_FAT*.png"), "_FAT"))

config_loader loaded from: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/src/config_loader.py


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


PatSeer Excel: 1639__dataset_08_06_26.xlsx  (1639 rows, 102 columns)
Columns:
  [  0] 'Record Number'
  [  1] 'Title'
  [  2] 'Abstract'
  [  3] 'Description of Drawings'
  [  4] 'CPC'
  [  5] 'PDF Link'
  [  6] 'Record Type'
  [  7] 'Publication/Issue Date'
  [  8] 'Filing/Application Date'
  [  9] 'Estimated Expiry Date'
  [ 10] 'EFAM Earliest Priority Date'
  [ 11] 'EFAM Earliest Publication Date'
  [ 12] 'Priority Details'
  [ 13] 'Priority Dates (All)'
  [ 14] 'Application No.'
  [ 15] 'Priority Country Code'
  [ 16] 'Priority Year'
  [ 17] 'Register Legal Status'
  [ 18] 'Register Legal Status Date'
  [ 19] 'Summary of Invention'
  [ 20] 'Designated States'
  [ 21] 'Active in Designated States'
  [ 22] 'Field of search'
  [ 23] 'Industry'
  [ 24] 'Tech Domain'
  [ 25] 'Tech Sub Domain'
  [ 26] 'Description'
  [ 27] 'Claims'
  [ 28] 'Number Of Claims'
  [ 29] 'No. of Independent Claims'
  [ 30] 'Independent Claims'
  [ 31] 'First Claim'
  [ 32] 'Advantages of Invention'
  [ 33] 'N

In [2]:
# ─── Dry-run on ONE patent — OCR preview, NO files changed ───────────────────
TEST_PATENT = "US2022234745A1"   # any Excel key or folder name

raw_dir    = cfg["paths"]["raw_images"]
folder_map = _build_folder_map(raw_dir)
img_dir    = folder_map.get(_patent_core(TEST_PATENT))

if img_dir is None:
    print(f"⚠  No folder found for {TEST_PATENT}")
    print(f"   Available: {sorted(folder_map.keys())}")
else:
    actual_id = img_dir.name
    figures   = _collect_figures(img_dir, actual_id)

    print(f"Excel key : {TEST_PATENT}  →  folder: {actual_id}")
    print(f"Figures   : {len(figures)}")
    print()
    for f in figures:
        print(f"  {f.name}")
    print()
    print("OCR will run on each file above and rename to _F001.png / _Fu001.png.")

Excel key : US2022234745A1  →  folder: US20220234745A1
Figures   : 11

  US20220234745A1_D00000.png
  US20220234745A1_D00001.png
  US20220234745A1_D00002.png
  US20220234745A1_D00003.png
  US20220234745A1_D00004.png
  US20220234745A1_D00005.png
  US20220234745A1_D00006.png
  US20220234745A1_D00007.png
  US20220234745A1_D00008.png
  US20220234745A1_D00009.png
  US20220234745A1_D00010.png

OCR will run on each file above and rename to _F001.png / _Fu001.png.


In [3]:
# ─── Full run — OCR every figure and rename ───────────────────────────────────
# Each image is read as-is (no splitting). OCR reads the "FIG. N" printed on
# the drawing and renames the file to _F001.png / _Fu001.png accordingly.
# Safe to re-run — already-labeled files (_F* / _Fu*) are skipped.

raw_dir    = cfg["paths"]["raw_images"]
folder_map = _build_folder_map(raw_dir)
totals     = {"labeled": 0, "unlabeled": 0, "patents": 0}
errors     = []

for patent_id in excel_index:
    img_dir = folder_map.get(_patent_core(patent_id))
    if img_dir is None:
        continue

    actual_id = img_dir.name
    figures   = _collect_figures(img_dir, actual_id)
    if not figures:
        continue

    alias = f" ({patent_id})" if actual_id != patent_id else ""
    print(f"\n── {actual_id}{alias}  ({len(figures)} files)")

    try:
        final_paths = ocr_and_rename_crops(actual_id, figures, cfg)
        n_labeled   = sum(1 for p in final_paths if "_Fu" not in p.name)
        n_unlabeled = len(final_paths) - n_labeled
        totals["labeled"]   += n_labeled
        totals["unlabeled"] += n_unlabeled
        totals["patents"]   += 1
    except Exception as exc:
        errors.append((actual_id, str(exc)))
        print(f"  ✗ {exc}")

print(f"\n{'═'*45}")
print(f"Patents processed : {totals['patents']}")
print(f"Labeled  (_F)     : {totals['labeled']}")
print(f"Unlabeled (_Fu)   : {totals['unlabeled']}")
if errors:
    print(f"Errors            : {len(errors)}")
    for pid, e in errors:
        print(f"  {pid}: {e}")
print(f"{'═'*45}")


── US11787551PAFP (US11787551B1)  (11 files)
    OCR  US11787551PAFP_img1.png  →  —
    OCR  US11787551PAFP_img2.png  →  —
    OCR  US11787551PAFP_img3.png  →  —
    OCR  US11787551PAFP_img4.png  →  —
    OCR  US11787551PAFP_img5.png  →  —
    OCR  US11787551PAFP_img6.png  →  —
    OCR  US11787551PAFP_img7.png  →  —
    OCR  US11787551PAFP_img8.png  →  —
    OCR  US11787551PAFP_img9.png  →  —
    OCR  US11787551PAFP_img10.png  →  —
    OCR  US11787551PAFP_img11.png  →  —
  Renamed 11 crops  (0 labeled  11 unlabeled)

── US20220234745A1 (US2022234745A1)  (11 files)
    OCR  US20220234745A1_D00000.png  →  —
    OCR  US20220234745A1_D00001.png  →  —
    OCR  US20220234745A1_D00002.png  →  —
    OCR  US20220234745A1_D00003.png  →  —
    OCR  US20220234745A1_D00004.png  →  —
    OCR  US20220234745A1_D00005.png  →  —
    OCR  US20220234745A1_D00006.png  →  —
    OCR  US20220234745A1_D00007.png  →  —
    OCR  US20220234745A1_D00008.png  →  —
    OCR  US20220234745A1_D00009.png  →  —
    OCR 

/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


    OCR  US2020148347A1_img1.png  →  —
    OCR  US2020148347A1_img2.png  →  —


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


    OCR  US2020148347A1_img3.png  →  —
    OCR  US2020148347A1_img4.png  →  —


...

/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


    OCR  US2020172235A1_img1.png  →  —
    OCR  US2020172235A1_img2.png  →  —


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


    OCR  US2020172235A1_img3.png  →  —
    OCR  US2020172235A1_img4.png  →  —
  Renamed 4 crops  (0 labeled  4 unlabeled)

═════════════════════════════════════════════
Patents processed : 8
Labeled  (_F)     : 0
Unlabeled (_Fu)   : 81
═════════════════════════════════════════════


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


In [ ]:
# ─── Matching report ─────────────────────────────────────────────────────────
if not all_summary:
    print("No results yet — run Cell 3 first.")
else:
    import pandas as pd

    df = pd.DataFrame(all_summary)

    total_patents   = len(df)
    perfect         = int((df["renamed_Fu"] == 0).sum())
    needs_review    = int((df["renamed_Fu"] > 0).sum())
    total_F         = int(df["renamed_F"].sum())
    total_Fu        = int(df["renamed_Fu"].sum())
    total_figs      = total_F + total_Fu
    kept_originals  = int(df["kept_originals"].sum())

    def _pct(n):
        return f"{n / total_patents * 100:.0f}%" if total_patents else "N/A"

    print("══════════════════════════════════════")
    print("Figure Matching Report")
    print("══════════════════════════════════════")
    print(f"Patents processed       : {total_patents}")
    print(f"Perfect matches (_F)    : {perfect}  ({_pct(perfect)})")
    print(f"Needs review (_Fu)      : {needs_review}  ({_pct(needs_review)})")
    print(f"Total figures matched   : {total_figs}")
    print(f"  - labeled   (_F)      : {total_F}")
    print(f"  - unlabeled (_Fu)     : {total_Fu}")
    print(f"D/FAT originals kept    : {kept_originals}")
    print(f"Run errors              : {len(run_errors)}")
    print("══════════════════════════════════════")

    if run_errors:
        print("\nErrors:")
        for pid, err in run_errors:
            print(f"  {pid}: {err}")